our company manages a loyality program for customers. we need to generate monthly statements that calculate
the total points earned by each customer based on their purchases.points are awarded as follows: 
1 point per dollar spent. 
Additionally,if a customer spends over $500 in a month,they receive a bonus of 50 points. 
write a simple and under stable PL\SQL procedure in detail 
explaining each code step in detail that calculates and updates the total loyality points for each customer 
for a given month. 
the procedure should use control statements to handle the logic for calculating the points,including 
the bonus for spending over $500 using below customers and purchases tables as reference 
-- create table customers ( customerid int primary key, customername varchar(100), 
totalpoints int) ; 
create table purchases(purchaseid int primary key, customerid int,purchase amount decimal(10,2), 
purchasedate date, constraints fk_customer FOREIGN KEY(customerid) references customers(customerid) 
- give me best performance oriented simple and also alternative solutions for the above queston?

For this problem, we can write a PL/SQL stored procedure that calculates loyalty points for each customer for a given month, adds a 50-point bonus when monthly spending exceeds $500, and updates the `totalpoints` column in the `customers` table.

I'll show you the simple, beginner-friendly solution first, followed by a more performance-oriented alternative.

## 1. Understand the business logic

The rules are:

- Earn 1 point for every $1 spent during the given month.
- If the customer's total monthly spending is strictly greater than $500, award an additional 50 points.
- Update the customer's `totalpoints` column.

For example:

| Customer | Monthly spending | Base points | Bonus | Total points earned |
| -------- | ---------------- | ----------- | ----- | ------------------- |
| Rahul    | $300             | 300         | 0     | 300                 |
| Priya    | $500             | 500         | 0     | 500                 |
| Amit     | $650             | 650         | 50    | 700                 |
| Sneha    | $800             | 800         | 50    | 850                 |

I'm assuming that `totalpoints` means the points earned for the specified month, so the procedure replaces the existing value. If you want to accumulate points across months, I'll show that adjustment too.

## 2. Create the tables in Oracle

Your table definitions need a small Oracle syntax adjustment: use `NUMBER(10,2)` for decimal amounts and `purchase_amount` as the column name.

```
CREATE TABLE customers (
    customerid   NUMBER PRIMARY KEY,
    customername VARCHAR2(100),
    totalpoints  NUMBER DEFAULT 0
);

CREATE TABLE purchases (
    purchaseid     NUMBER PRIMARY KEY,
    customerid     NUMBER,
    purchase_amount NUMBER(10,2),
    purchasedate   DATE,
    CONSTRAINT fk_customer
        FOREIGN KEY (customerid)
        REFERENCES customers(customerid)
);
```

## 3. Solution 1 — Simple PL/SQL procedure using a FOR LOOP and IF

This solution is best for learning PL/SQL control statements. It calculates each customer's monthly spending, applies the bonus condition, and updates their points.

```

CREATE OR REPLACE PROCEDURE CalculateLoyaltyPoints (
    p_month IN DATE
)
AS
    v_month_start DATE;
    v_month_end   DATE;
    v_spending    NUMBER;
    v_points      NUMBER;
BEGIN
    -- Get the first day of the requested month
    v_month_start := TRUNC(p_month, 'MM');

    -- Get the first day of the next month
    v_month_end := ADD_MONTHS(v_month_start, 1);

    -- Process each customer
    FOR cust IN (
        SELECT c.customerid,
               NVL(SUM(p.purchase_amount), 0) AS monthly_spending
        FROM customers c
        LEFT JOIN purchases p
            ON c.customerid = p.customerid
           AND p.purchasedate >= v_month_start
           AND p.purchasedate < v_month_end
        GROUP BY c.customerid
    )
    LOOP
        -- Get monthly spending
        v_spending := cust.monthly_spending;

        -- One point per whole dollar spent
        v_points := TRUNC(v_spending);

        -- Bonus points for spending over $500
        IF v_spending > 500 THEN
            v_points := v_points + 50;
        END IF;

        -- Update the customer's points
        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = cust.customerid;

    END LOOP;

    COMMIT;
END;
/

```

### How the procedure works

Step 1 — Accept the month

```
p_month IN DATE
```

The procedure accepts a date representing the month you want to process. For example, you can pass any date in March 2026.

Step 2 — Define the date range

```
v_month_start := TRUNC(p_month, 'MM');
v_month_end := ADD_MONTHS(v_month_start, 1);
```

If the input is March 15, 2026:

- `v_month_start` becomes March 1, 2026.
- `v_month_end` becomes April 1, 2026.

We use a half-open date range, including the start and excluding the end, so purchases on March 31 are included even if they have a time component. This also avoids applying a function to the `purchasedate` column in the filter, which can help Oracle use a suitable index.&#x20;

[image](https://www.google.com/s2/favicons?domain=https://asktom.oracle.com\&sz=32)

Oracle Magazine

+1



Step 3 — Calculate monthly spending

```
FOR cust IN (
    SELECT c.customerid,
           NVL(SUM(p.purchase_amount), 0) AS monthly_spending
    FROM customers c
    LEFT JOIN purchases p
        ON c.customerid = p.customerid
       AND p.purchasedate >= v_month_start
       AND p.purchasedate < v_month_end
    GROUP BY c.customerid
)
LOOP
```

This query calculates total spending for every customer during the requested month.

- `SUM(p.purchase_amount)` adds all qualifying purchases.
- `NVL(..., 0)` returns zero if the customer has no purchases.
- `LEFT JOIN` ensures customers without purchases are included.
- `GROUP BY c.customerid` produces one result per customer.
- `FOR cust IN (...) LOOP` processes those results one at a time.

Step 4 — Calculate the base points

```
v_points := TRUNC(v_spending);
```

We use `TRUNC` because `totalpoints` is a whole-number column and the rule awards one point per whole dollar. For example, $250.75 earns 250 base points under this assumption.

Step 5 — Apply the bonus using `IF`

```
IF v_spending > 500 THEN
    v_points := v_points + 50;
END IF;
```

The condition must be strictly greater than $500.

- Spending of $500 earns no bonus.
- Spending of $500.01 earns the additional 50 points.

Step 6 — Update the customer

```
UPDATE customers
SET totalpoints = v_points
WHERE customerid = cust.customerid;
```

Only the current customer's record is updated.

Step 7 — Commit

```
COMMIT;
```

Saves all updates after the loop has processed the customers.

### Execute the procedure

```
BEGIN
    CalculateLoyaltyPoints(DATE '2026-03-01');
END;
/
```

Check the results:

```
SELECT customerid, customername, totalpoints
FROM customers
ORDER BY customerid;
```

## 4. Solution 2 — Performance-oriented alternative

For a large number of customers, you can avoid the PL/SQL `FOR LOOP` and individual `UPDATE` statements by using a single set-based `UPDATE` statement.

This version uses SQL's `CASE` expression instead of PL/SQL's `IF` statement.

```

CREATE OR REPLACE PROCEDURE CalculateLoyaltyPoints_Fast (
    p_month IN DATE
)
AS
    v_month_start DATE;
    v_month_end   DATE;
BEGIN
    v_month_start := TRUNC(p_month, 'MM');
    v_month_end   := ADD_MONTHS(v_month_start, 1);

    UPDATE customers c
    SET totalpoints = (
        SELECT TRUNC(NVL(SUM(p.purchase_amount), 0))
               + CASE
                   WHEN NVL(SUM(p.purchase_amount), 0) > 500
                   THEN 50
                   ELSE 0
                 END
        FROM purchases p
        WHERE p.customerid = c.customerid
          AND p.purchasedate >= v_month_start
          AND p.purchasedate < v_month_end
    );

    COMMIT;
END;
/

```

Execute it in the same way:

```
BEGIN
    CalculateLoyaltyPoints_Fast(DATE '2026-03-01');
END;
/
```

### Why is this alternative useful?

- It uses one SQL `UPDATE` statement instead of issuing an update for every customer from a PL/SQL loop.
- `SUM()` calculates the customer's monthly spending.
- `CASE` applies the additional 50 points when spending exceeds $500.
- Customers without purchases receive zero points.
- Oracle can optimize the set-based statement, although actual performance depends on table size, indexes, and the execution plan.

For a large dataset, consider an index on `purchases(customerid, purchasedate)` and evaluate the execution plan against your real data. The best index depends on the workload.

## 5. Compare the two solutions

| Feature                     | Solution 1: Loop + IF                 | Solution 2: Single UPDATE          |
| --------------------------- | ------------------------------------- | ---------------------------------- |
| Bonus logic                 | PL/SQL `IF`                           | SQL `CASE`                         |
| Updates                     | One per customer                      | One set-based statement            |
| Easy for beginners          | Yes                                   | Yes, once SQL is familiar          |
| Large datasets              | May incur more PL/SQL-to-SQL overhead | Often a better starting point      |
| Control statements practice | Excellent                             | Demonstrates SQL conditional logic |

My recommendation: Learn Solution 1 first because it demonstrates the PL/SQL control statements in your question. For production-scale processing, start by evaluating Solution 2.

One final business-rule distinction: both procedures replace `totalpoints` with the points earned for the selected month. If `totalpoints` is intended to be a lifetime balance, you should instead add the newly calculated monthly points to the existing balance, while ensuring the same month cannot be credited twice.

note: DEFAULT 0 means: if you don’t give a value for that column when inserting a row, 
Oracle automatically stores 0.
NUMBER(10,2) means a number with up to 10 digits in total, of which 2 are after the decimal point.
10 = precision: the total number of digits allowed.
2 = scale: how many of those digits come after the decimal point.
NUMBER(10,2) means a number with up to 10 digits in total, of which 2 are after the decimal point.

10 = precision: the total number of digits allowed.
2 = scale: how many of those digits come after the decimal point.

**TRUNC** is an Oracle function that **cuts off (truncates) part of a value** 
instead of rounding it. It works on both dates and numbers.

## 1. TRUNC with dates

It chops a date back to a chosen unit, and the rest is reset.

```sql
TRUNC(date)            -- removes the time, keeps the day
TRUNC(date, 'MM')      -- first day of the month
TRUNC(date, 'YYYY')    -- first day of the year
```

| Expression | Result |
|---|---|
| `TRUNC(SYSDATE)` | Today at 00:00:00 |
| `TRUNC(DATE '2025-08-15', 'MM')` | `01-AUG-2025` |
| `TRUNC(DATE '2025-08-15', 'YYYY')` | `01-JAN-2025` |

## 2. TRUNC with numbers

It drops digits after a given decimal place **without rounding**.

```sql
TRUNC(number, places)
```

| Expression | Result |
|---|---|
| `TRUNC(125.789)` | `125` |
| `TRUNC(125.789, 2)` | `125.78` |
| `TRUNC(125.789, 1)` | `125.7` |
| `TRUNC(125.789, -1)` | `120` |

## TRUNC vs ROUND

- `TRUNC(125.789, 2)` gives `125.78` (just cuts off)
- `ROUND(125.789, 2)` gives `125.79` (rounds up)

In your line `TRUNC(p_month, 'MM')`, it's the date version: it cuts the date back to 
the first day of that month.

`ADD_MONTHS(date, n)` is an Oracle function that **adds `n` months to a date** and returns the new date.

In your line, `ADD_MONTHS(v_month_start, 1)` adds 1 month to `v_month_start`, so you get the **first 
day of the next month** (because `v_month_start` is already the 1st).

**Examples:**

| Expression | Result |
|---|---|
| `ADD_MONTHS(DATE '2025-08-01', 1)` | `01-SEP-2025` |
| `ADD_MONTHS(DATE '2025-12-01', 1)` | `01-JAN-2026` |
| `ADD_MONTHS(DATE '2025-08-01', -1)` | `01-JUL-2025` (negative goes back) |
| `ADD_MONTHS(DATE '2025-01-31', 1)` | `28-FEB-2025` (clamps to the last day of the month) |

**Why it's useful:** it handles different month lengths (28, 30, 31 days) and year changes for you, 
so you don't have to calculate them yourself.

**Together with TRUNC:**

```sql
v_month_start := TRUNC(p_month, 'MM');           -- 01-AUG-2025
v_month_end   := ADD_MONTHS(v_month_start, 1);   -- 01-SEP-2025
```

Then you can select everything from August with:

```sql
WHERE purchase_date >= v_month_start
  AND purchase_date <  v_month_end
```

**Note:** `ADD_MONTHS(...)` by itself, ending with `;`, is not a complete statement. 
It must be assigned to a variable (`v_month_end := ...;`) or used inside a 
`SELECT`, like `SELECT ADD_MONTHS(SYSDATE, 1) FROM dual;`.